# Ноутбук 03a — ML-модели: XGBoost, Random Forest, Elastic Net
**Подраздел 3.2 ПЗ** — подбор гиперпараметров  
**Подраздел 3.3 ПЗ** — результаты прогнозирования (горизонты 1, 3, 6, 12 недель)

Порядок запуска: **после** `02_stationarity_acf_pacf_adf.ipynb`.  
Зависимости: `data/processed/features_train.parquet`, `data/processed/features_test.parquet`.

Артефакты:
- `models/saved/xgboost_h{1,3,6,12}.pkl`  
- `models/saved/rf_h{1,3,6,12}.pkl`  
- `models/saved/elasticnet_h{1,3,6,12}.pkl`  
- `reports/tables/table_3_metrics_ml.csv` — RMSE, MAE, MAPE по горизонтам  
- `reports/figures/fig_3_forecast_xgb_h1.png` — прогноз vs факт (горизонт 1)  
- `reports/figures/fig_3_forecast_rf_h1.png`  
- `reports/figures/fig_3_forecast_en_h1.png`  


In [2]:
import sys, warnings, pickle
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

from src.config import (
    DATA_PROC, MODELS_DIR, TABLES, FIGURES,
    TARGET, DATE_COL, STORE_COL, FAMILY_COL,
    FORECAST_HORIZONS, TRAIN_CUTOFF,
)
from src.evaluation.backtesting import make_horizon_target, get_feature_cols
from src.evaluation.metrics import compute_metrics, metrics_table
from src.models.ml_models import fit_xgboost, fit_random_forest, fit_elasticnet_cv
from src.features.scaling import apply_standard_scaler

print("Импорты выполнены.")
print("Горизонты прогноза:", FORECAST_HORIZONS)


Импорты выполнены.
Горизонты прогноза: [1, 3, 6, 12]


## Ячейка 1 — Загрузка признакового пространства

In [3]:
df_train = pd.read_parquet(DATA_PROC / "features_train.parquet")
df_test  = pd.read_parquet(DATA_PROC / "features_test.parquet")

# Объединяем для корректного конструирования target_h (нужен сдвиг в будущее)
df_all = pd.concat([df_train, df_test], ignore_index=True).sort_values(
    [STORE_COL, FAMILY_COL, DATE_COL]
).reset_index(drop=True)

print(f"Обучающая выборка: {df_train.shape}")
print(f"Тестовая выборка:  {df_test.shape}")
print(f"Объединённый датафрейм: {df_all.shape}")
print(f"\nСтолбцы (первые 10): {list(df_all.columns[:10])}")
print(f"Диапазон дат: {df_all[DATE_COL].min().date()} — {df_all[DATE_COL].max().date()}")


Обучающая выборка: (2872584, 31)
Тестовая выборка:  (128304, 31)
Объединённый датафрейм: (3000888, 31)

Столбцы (первые 10): ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster']
Диапазон дат: 2013-01-01 — 2017-08-15


## Ячейка 2 — Определение признаковых столбцов

Список признаков определяется функцией `get_feature_cols()` из модуля `backtesting.py`.  
Она исключает служебные столбцы (date, store_nbr, family), сырые целевые переменные  
и вспомогательные промежуточные столбцы.  

XGBoost и Random Forest обучаются на **ненормализованных** признаках (инвариантны к масштабу).  
Elastic Net обучается на **нормализованных** признаках (StandardScaler из ячейки 2.4.6 NB01).


In [6]:
from sklearn.preprocessing import LabelEncoder

# Определяем категориальные столбцы среди признаков
cat_cols = [c for c in FEATURE_COLS
            if df_train[c].dtype == object or str(df_train[c].dtype) == 'category']

print("Категориальные столбцы для кодирования:", cat_cols)

# Label-кодирование: обучаем на всём датасете, применяем везде
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([df_train[col], df_test[col]]).astype(str))
    df_train[col] = le.transform(df_train[col].astype(str))
    df_test[col]  = le.transform(df_test[col].astype(str))
    df_all[col]   = le.transform(df_all[col].astype(str))
    le_dict[col]  = le

print("Кодирование выполнено.")

# Теперь нормализация для ElasticNet
df_train_scaled, df_test_scaled, scaler = apply_standard_scaler(
    df_train, df_test, FEATURE_COLS
)
print("Нормализация для ElasticNet выполнена.")
FEATURE_COLS = get_feature_cols(df_all)
print(f"Число признаков: {len(FEATURE_COLS)}")
print("Признаки:", FEATURE_COLS)

# Нормализованный вариант для ElasticNet (fit только на train)
df_train_scaled, df_test_scaled, scaler = apply_standard_scaler(
    df_train, df_test, FEATURE_COLS
)
print("\nНормализация для ElasticNet выполнена.")
print(f"  mean(lag_1) после нормализации (train): {df_train_scaled['lag_1'].mean():.4f}")

FEATURE_COLS = get_feature_cols(df_all)
print(f"Число признаков: {len(FEATURE_COLS)}")
print("Признаки:", FEATURE_COLS)

# Нормализованный вариант для ElasticNet (fit только на train)
df_train_scaled, df_test_scaled, scaler = apply_standard_scaler(
    df_train, df_test, FEATURE_COLS
)
print("\nНормализация для ElasticNet выполнена.")
print(f"  mean(sales_lag_1w) после нормализации (train): {df_train_scaled['sales_lag_1w'].mean():.4f}")


Категориальные столбцы для кодирования: []
Кодирование выполнено.
Нормализация для ElasticNet выполнена.
Число признаков: 28
Признаки: ['id', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'transactions', 'is_holiday', 'sales_diff52w', 'year', 'month', 'week', 'day_of_week', 'day_of_year', 'week_sin', 'week_cos', 'sales_lag_1w', 'sales_lag_2w', 'sales_lag_4w', 'sales_lag_9w', 'sales_lag_12w', 'sales_lag_52w', 'sales_roll_4w_mean', 'sales_roll_4w_std', 'sales_roll_12w_mean', 'sales_roll_12w_std']


MemoryError: Unable to allocate 614. MiB for an array with shape (2872584, 28) and data type float64

## Ячейка 3 — Обучение моделей по горизонтам (direct forecasting)

**Стратегия прямого прогнозирования (direct multi-step):**  
Для каждого горизонта h ∈ {1, 3, 6, 12} недель конструируется отдельная целевая переменная:  
`target_h[t] = log1p(sales_weekly[t + h])`  

Признаки в строке t описывают состояние на момент t (lag_1 = y(t−1) и т.д.).  
Следовательно, для h=1 модель предсказывает следующую неделю,  
для h=12 — значение через 12 недель, используя те же входные признаки момента t.

**Преимущество прямого подхода перед рекурсивным:**  
ошибка не накапливается через цепочку промежуточных прогнозов.


In [ ]:
results_xgb = {}
results_rf  = {}
results_en  = {}
models_xgb  = {}
models_rf   = {}
models_en   = {}

cutoff = pd.Timestamp(TRAIN_CUTOFF)

for h in FORECAST_HORIZONS:
    print(f"\n{'='*60}")
    print(f"Горизонт h = {h} нед.")
    print(f"{'='*60}")

    # ── Конструирование target_h ──────────────────────────────────────────
    df_h = make_horizon_target(df_all, horizon=h)
    target_col = f"target_h{h}"

    # Разбивка по дате отсечения
    train_h = df_h[df_h[DATE_COL] < cutoff].dropna(subset=[target_col])
    test_h  = df_h[df_h[DATE_COL] >= cutoff].dropna(subset=[target_col])

    X_train = train_h[FEATURE_COLS].fillna(0)
    y_train = train_h[target_col]
    X_test  = test_h[FEATURE_COLS].fillna(0)
    y_test  = test_h[target_col]

    print(f"  Обучающая: {len(train_h):,} строк | Тестовая: {len(test_h):,} строк")

    # ── XGBoost ────────────────────────────────────────────────────────────
    print("  [XGBoost] обучение...")
    xgb_model, _ = fit_xgboost(X_train, y_train)
    xgb_pred = xgb_model.predict(X_test)
    metrics_xgb = compute_metrics(y_test.values, xgb_pred, log_scale=True)
    results_xgb[h] = metrics_xgb
    models_xgb[h]  = xgb_model
    print(f"  XGBoost  → RMSE={metrics_xgb['RMSE']:.2f}, MAE={metrics_xgb['MAE']:.2f}, MAPE={metrics_xgb['MAPE']:.2f}%")

    # ── Random Forest ──────────────────────────────────────────────────────
    print("  [Random Forest] обучение...")
    rf_model = fit_random_forest(X_train, y_train)
    rf_pred  = rf_model.predict(X_test)
    metrics_rf = compute_metrics(y_test.values, rf_pred, log_scale=True)
    results_rf[h] = metrics_rf
    models_rf[h]  = rf_model
    print(f"  RF       → RMSE={metrics_rf['RMSE']:.2f}, MAE={metrics_rf['MAE']:.2f}, MAPE={metrics_rf['MAPE']:.2f}%")

    # ── Elastic Net (нормализованные признаки) ─────────────────────────────
    print("  [Elastic Net] обучение...")
    # Пересчёт нормализованных признаков с target_h
    df_h_scaled = df_h.copy()
    # Применяем тот же scaler к объединённому датафрейму с target_h
    df_h_scaled[FEATURE_COLS] = scaler.transform(df_h[FEATURE_COLS].fillna(0))
    train_en = df_h_scaled[df_h_scaled[DATE_COL] < cutoff].dropna(subset=[target_col])
    test_en  = df_h_scaled[df_h_scaled[DATE_COL] >= cutoff].dropna(subset=[target_col])
    X_train_en = train_en[FEATURE_COLS].fillna(0)
    y_train_en = train_en[target_col]
    X_test_en  = test_en[FEATURE_COLS].fillna(0)
    y_test_en  = test_en[target_col]

    en_model = fit_elasticnet_cv(X_train_en, y_train_en)
    en_pred  = en_model.predict(X_test_en)
    metrics_en = compute_metrics(y_test_en.values, en_pred, log_scale=True)
    results_en[h] = metrics_en
    models_en[h]  = en_model
    print(f"  ElasticNet → RMSE={metrics_en['RMSE']:.2f}, MAE={metrics_en['MAE']:.2f}, MAPE={metrics_en['MAPE']:.2f}%")

print("\n[OK] Обучение всех ML-моделей завершено.")


## Ячейка 4 — Сохранение обученных моделей

In [ ]:
import pickle

for h in FORECAST_HORIZONS:
    with open(MODELS_DIR / f"xgboost_h{h}.pkl", "wb") as f:
        pickle.dump(models_xgb[h], f)
    with open(MODELS_DIR / f"rf_h{h}.pkl", "wb") as f:
        pickle.dump(models_rf[h], f)
    with open(MODELS_DIR / f"elasticnet_h{h}.pkl", "wb") as f:
        pickle.dump(models_en[h], f)

print("Модели сохранены в models/saved/:")
for h in FORECAST_HORIZONS:
    print(f"  xgboost_h{h}.pkl, rf_h{h}.pkl, elasticnet_h{h}.pkl")


## Ячейка 5 — Сводная таблица метрик ML-моделей

**Таблица 3.10 ПЗ** (фрагмент — ML-модели): RMSE, MAE, MAPE на тестовой выборке  
по горизонтам h ∈ {1, 3, 6, 12} недель.


In [ ]:
summary = {}
for name, res in [("XGBoost", results_xgb), ("RandomForest", results_rf), ("ElasticNet", results_en)]:
    summary[name] = {}
    for h in FORECAST_HORIZONS:
        summary[name][h] = res[h]

df_summary = metrics_table(summary)
print("Сводная таблица метрик ML-моделей:")
print(df_summary.to_string())

# Сохранение
df_summary.to_csv(TABLES / "table_3_metrics_ml.csv")
print("\nСохранено: reports/tables/table_3_metrics_ml.csv")


## Ячейка 6 — Графики прогноз vs факт (горизонт h=1)

**Рисунки 3.2–3.4 ПЗ**: визуальное сравнение прогноза с фактическими продажами  
на агрегированном недельном ряде (сумма по всем магазинам и семействам).

Горизонт h=1 визуализируется первым как наиболее точный прогноз;  
горизонты 3, 6, 12 добавляются в ноутбуке 05.


In [ ]:
def plot_forecast_vs_actual(
    df_all, models, scaler_obj, feature_cols, h, model_name, fig_path,
    use_scaling=False
):
    """Строит график прогноз vs факт для агрегированного ряда."""
    from src.evaluation.backtesting import make_horizon_target
    cutoff = pd.Timestamp(TRAIN_CUTOFF)
    target_col = f"target_h{h}"

    df_h = make_horizon_target(df_all, horizon=h)
    test_h = df_h[df_h[DATE_COL] >= cutoff].dropna(subset=[target_col])

    if use_scaling:
        X_test = pd.DataFrame(
            scaler_obj.transform(test_h[feature_cols].fillna(0)),
            columns=feature_cols
        )
    else:
        X_test = test_h[feature_cols].fillna(0)

    pred_log = models[h].predict(X_test)
    y_true   = np.expm1(test_h[target_col].values)
    y_pred   = np.expm1(np.clip(pred_log, -1e6, 20))

    # Агрегируем по дате (сумма по магазинам и семействам)
    agg = pd.DataFrame({"date": test_h[DATE_COL].values, "actual": y_true, "pred": y_pred})
    agg = agg.groupby("date").sum().reset_index()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(agg["date"], agg["actual"], label="Факт", linewidth=1.5, color="steelblue")
    ax.plot(agg["date"], agg["pred"],   label=f"Прогноз {model_name}", linewidth=1.5,
            color="tomato", linestyle="--")
    ax.set_title(f"Рисунок 3 — {model_name}: прогноз vs факт (h={h} нед.)")
    ax.set_xlabel("Дата")
    ax.set_ylabel("Суммарные продажи, ед.")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_path, dpi=120)
    plt.show()
    print(f"Сохранено: {fig_path}")

h1 = 1
plot_forecast_vs_actual(df_all, models_xgb, scaler, FEATURE_COLS, h1, "XGBoost",
                         FIGURES / "fig_3_forecast_xgb_h1.png", use_scaling=False)
plot_forecast_vs_actual(df_all, models_rf, scaler, FEATURE_COLS, h1, "RandomForest",
                         FIGURES / "fig_3_forecast_rf_h1.png", use_scaling=False)
plot_forecast_vs_actual(df_all, models_en, scaler, FEATURE_COLS, h1, "ElasticNet",
                         FIGURES / "fig_3_forecast_en_h1.png", use_scaling=True)


## Ячейка 7 — Важность признаков XGBoost (горизонт h=1)

In [ ]:
model_h1 = models_xgb[1]
importances = pd.Series(model_h1.feature_importances_, index=FEATURE_COLS)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
top15.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Рисунок 3.x — Важность признаков XGBoost (h=1 нед.)")
ax.set_xlabel("Feature importance (gain)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(FIGURES / "fig_3_xgb_feature_importance_h1.png", dpi=120)
plt.show()
print("Топ-5 признаков по важности:")
print(top15.head(5).to_string())


## Ячейка 8 — Анализ остатков XGBoost (горизонт h=1)

**Рисунок 3.8 ПЗ**: распределение остатков и автокорреляция.  
Если остатки не имеют структуры (тест Льюнга-Бокса не отвергает H₀),  
модель адекватно захватила систематическую часть ряда.


In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

h_res = 1
df_h  = make_horizon_target(df_all, horizon=h_res)
target_col = f"target_h{h_res}"
cutoff = pd.Timestamp(TRAIN_CUTOFF)
test_h = df_h[df_h[DATE_COL] >= cutoff].dropna(subset=[target_col])

X_test = test_h[FEATURE_COLS].fillna(0)
pred   = models_xgb[h_res].predict(X_test)
# Остатки на исходной шкале
residuals = np.expm1(test_h[target_col].values) - np.expm1(np.clip(pred, -1e6, 20))

# Агрегируем по дате для анализа ряда остатков
res_series = pd.Series(residuals, index=test_h[DATE_COL].values)
res_agg = res_series.groupby(level=0).mean().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(res_agg, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Распределение остатков")
axes[0].set_xlabel("Остаток, ед.")
axes[0].axvline(0, color="red", linestyle="--")

axes[1].plot(res_agg.index, res_agg.values, linewidth=0.8, color="steelblue")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Остатки во времени")
axes[1].set_xlabel("Дата")

from statsmodels.graphics.tsaplots import plot_acf
plot_acf(res_agg.dropna(), lags=20, ax=axes[2], alpha=0.05)
axes[2].set_title("АКФ остатков (лаги 1–20)")

plt.suptitle("Рисунок 3.8 — Диагностика остатков XGBoost (h=1 нед.)")
plt.tight_layout()
plt.savefig(FIGURES / "fig_3_residuals_xgb_h1.png", dpi=120)
plt.show()

# Тест Льюнга-Бокса
lb = acorr_ljungbox(res_agg.dropna(), lags=[10, 20], return_df=True)
print("Тест Льюнга-Бокса (H₀: нет автокорреляции):")
print(lb.to_string())
_, pnorm = stats.normaltest(res_agg.dropna())
print(f"\nТест нормальности (D'Agostino-Pearson): p = {pnorm:.4f}")
if pnorm > 0.05:
    print("  → H₀ нормальности не отвергается.")
else:
    print("  → Остатки отклоняются от нормального распределения.")


In [ ]:
print("=" * 60)
print("Ноутбук 03a выполнен.")
print("=" * 60)
print("Артефакты:")
for h in FORECAST_HORIZONS:
    print(f"  models/saved/xgboost_h{h}.pkl")
    print(f"  models/saved/rf_h{h}.pkl")
    print(f"  models/saved/elasticnet_h{h}.pkl")
print("  reports/tables/table_3_metrics_ml.csv")
print("  reports/figures/fig_3_forecast_xgb_h1.png")
print("  reports/figures/fig_3_forecast_rf_h1.png")
print("  reports/figures/fig_3_forecast_en_h1.png")
print("  reports/figures/fig_3_xgb_feature_importance_h1.png")
print("  reports/figures/fig_3_residuals_xgb_h1.png")
